In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

# --- Config ---
users = ["U001", "U002", "U003", "U004", "U005"]
start_date = datetime(2024, 1, 1)
n_days = 60  # 60 days per user

exercise_types = ["Running", "Cycling", "Strength", "Yoga", "Walking", None]
sleep_qualities = ["Poor", "Fair", "Good", "Excellent"]

records = []

for user in users:
    # Give each user a "baseline" personality
    user_step_base = np.random.randint(4000, 11000)
    user_hr_base = np.random.randint(62, 78)
    user_sleep_base = np.random.uniform(5.5, 8.0)

    for d in range(n_days):
        date = start_date + timedelta(days=d)

        # Activity
        steps = max(0, int(np.random.normal(user_step_base, 2500)))
        active_minutes = max(0, int(steps / 120 + np.random.normal(0, 5)))
        calories = steps * 0.04 + np.random.normal(0, 20)

        # Heart rate
        hr = np.random.normal(user_hr_base + steps/2000, 6)

        # Sleep
        sleep_hours = np.random.normal(user_sleep_base, 0.9)
        sleep_quality = np.random.choice(sleep_qualities, p=[0.1, 0.25, 0.4, 0.25])

        # SpO2
        spo2 = np.random.normal(97.5, 1.2)

        # Exercise (happens ~50% of days)
        if np.random.rand() < 0.5:
            ex = np.random.choice(exercise_types[:5])
        else:
            ex = None

        # Timestamp (with inconsistent formats)
        ts = date + timedelta(hours=np.random.randint(6, 22),
                              minutes=np.random.randint(0, 60))
        fmt_choice = np.random.choice(["iso", "uk", "text"], p=[0.8, 0.1, 0.1])
        if fmt_choice == "iso":
            ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")
        elif fmt_choice == "uk":
            ts_str = ts.strftime("%d/%m/%Y %H:%M")
        else:
            ts_str = ts.strftime("%b %d, %Y %H:%M")

        records.append({
            "user_id": user,
            "date": date.strftime("%Y-%m-%d"),
            "timestamp": ts_str,
            "steps": steps,
            "heart_rate_bpm": round(hr, 1),
            "sleep_hours": round(sleep_hours, 2),
            "sleep_quality": sleep_quality,
            "exercise_type": ex,
            "calories_burned": round(calories, 1),
            "active_minutes": active_minutes,
            "spo2_percent": round(spo2, 1),
        })

df = pd.DataFrame(records)

# --- Inject realistic messiness ---

# 1. Missing values (device didn't sync)
for col, frac in [("steps", 0.05), ("heart_rate_bpm", 0.04),
                  ("sleep_hours", 0.06), ("spo2_percent", 0.03),
                  ("calories_burned", 0.03)]:
    idx = np.random.choice(df.index, size=int(len(df)*frac), replace=False)
    df.loc[idx, col] = np.nan

# 2. Impossible outliers
out_idx = np.random.choice(df.index, size=15, replace=False)
df.loc[out_idx[:5], "heart_rate_bpm"] = np.random.choice([0, 320, 450], 5)
df.loc[out_idx[5:10], "spo2_percent"] = np.random.choice([60, 150, 0], 5)
df.loc[out_idx[10:], "steps"] = np.random.choice([-500, 200000], 5)

# 3. Duplicate rows (device sync error)
dupes = df.sample(20, random_state=1)
df = pd.concat([df, dupes], ignore_index=True)

# 4. Inconsistent category labels
non_null_mask = df["exercise_type"].notna()
random_mask = non_null_mask & (np.random.rand(len(df)) < 0.15)
df.loc[random_mask, "exercise_type"] = df.loc[random_mask, "exercise_type"].str.lower()

# 5. Shuffle so duplicates aren't at the bottom
df = df.sample(frac=1, random_state=2).reset_index(drop=True)

df.to_csv("wearable_data.csv", index=False)
print(f"Created wearable_data.csv with {len(df)} rows and {len(df.columns)} columns")
print(df.head())

Created wearable_data.csv with 320 rows and 11 columns
  user_id        date            timestamp    steps  heart_rate_bpm  \
0    U003  2024-02-16  2024-02-16 21:31:00  12855.0            82.9   
1    U004  2024-01-11  2024-01-11 08:53:00   7121.0             0.0   
2    U001  2024-01-08  2024-01-08 19:36:00   4770.0            83.3   
3    U002  2024-01-06  2024-01-06 11:21:00  12956.0            67.2   
4    U002  2024-02-08  2024-02-08 13:07:00  14454.0            62.7   

   sleep_hours sleep_quality exercise_type  calories_burned  active_minutes  \
0         6.94          Good       Running            500.0             101   
1         6.26          Good           NaN            320.5              55   
2         7.41          Fair           NaN            138.4              47   
3         6.56     Excellent          Yoga            500.1             104   
4         5.98          Good       Cycling            585.5             127   

   spo2_percent  
0          97.6  
1      